# **Предобработка данных для анализа заболеваний сердца**

## **1.	Очистка данных**

### **1.1.	Обработка пропущенных значений**

* __Цель:__ Обеспечить целостность набора данных, исключив влияние пустых значений на статистические показатели и алгоритмы машинного обучения.
* __Задачи:__
  - Обнаружить и классифицировать пропуски.
  - Принять решение о стратегии их заполнения или удаления.
* __Алгоритм использования:__
  1. __Анализ:__ Определить количество и процент пропусков в каждом столбце.
  2. __Диагностика:__ Определить природу пропусков (случайные/системные).
  3. __Применение стратегии:__
     * __Удаление:__ Исключение из набора данных строк или столбцов (признаков), в которых доля пропусков превышает критический порог.
     * __Заполнение:__
       * __Константой__ (например, "Не указано", 0).
       * __Статистическими мерами:__
         * Числовые данные: Заполнение средним значением или медианой;
         * Категориальные данные: Заполнение модой (наиболее вероятное значение).
       * __Значением из следующей или предыдущей строки__ для временных рядов.

In [ ]:
# --- 1. ЗАГРУЗКА И ПЕРВИЧНАЯ НАСТРОЙКА ---

# Установка и загрузка необходимых библиотек
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown
from scipy.stats import shapiro
from sklearn.preprocessing import PowerTransformer, StandardScaler

# Конфигурация логирования
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [ ]:
# Загрузка набора данных
try:
    data = pd.read_csv("../data/raw/heart_disease_uci.csv")
    display(Markdown("#### **Содержание набора данных:**"))
    display(data.head())
    display(Markdown("#### **Структура набора данных:**"))
    data.info()
except FileNotFoundError:
    logging.error(
        "Файл '../data/raw/heart_disease_uci.csv' не найден. Убедитесь, что скрипт загрузки данных был запущен."
    )

In [ ]:
# --- 2. ФУНКЦИИ ДЛЯ АНАЛИЗА И ОБРАБОТКИ ---


def get_missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Создает сводную таблицу по пропущенным значениям."""
    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    missing_summary = pd.DataFrame(
        {"Missing Values": missing_count, "Percentage (%)": missing_percent}
    )
    missing_summary = missing_summary.sort_values(by="Percentage (%)", ascending=False)
    return missing_summary

In [ ]:
# --- 3. КОНФИГУРАЦИЯ И ОПРЕДЕЛЕНИЕ ТИПОВ ПРИЗНАКОВ ---

# Группировка признаков по типу
ALL_COLUMNS = data.columns
FEATURE_CONFIG = {
    "ordinal": ["restecg", "slope", "ca"],  # Категориальные признаки с порядком
    "binary": [
        "sex",
        "fbs",
        "exang",
    ],  # Категориальные признаки с двумя категориями (бинарные)
    "nominal": [
        "dataset",
        "cp",
        "thal",
    ],  # Категориальные признаки без порядка (номинальные)
    "numerical": [
        "age",
        "trestbps",
        "chol",
        "thalch",
        "oldpeak",
    ],  # Количественные признаки (числовые)
}

for group, columns in FEATURE_CONFIG.items():
    FEATURE_CONFIG[group] = [col for col in columns if col in ALL_COLUMNS]

In [ ]:
# --- 4. АНАЛИЗ ПРОПУСКОВ "ДО" ---

display(Markdown("#### **Анализ пропущенных значений ДО обработки**"))
missing_summary_before = get_missing_summary(data)
display(missing_summary_before)

In [ ]:
# Визуализация пропусков с помощью тепловой карты (heatmap)
plt.figure(figsize=(16, 6))
sns.heatmap(data.isnull(), cbar=False, yticklabels=False, cmap="cubehelix_r")
plt.title("Тепловая карта пропусков (heatmap)", fontsize=14)
plt.show()

In [ ]:
# --- 5. ПРИМЕНЕНИЕ СТРАТЕГИИ ---

display(Markdown("#### **Применение стратегии обработки пропущенных значений**"))

# Создание копии набора данных для дальнейшей обработки пропущенных значений
data_cleaned = data.copy()
THRESHOLD = 50  # Установка порога (50%)

# 1.  Удаление столбцов, в которых процент пропусков превышает установленный порог в 50%
cols_to_drop = missing_summary_before[
    missing_summary_before["Percentage (%)"] > THRESHOLD
].index

if not cols_to_drop.empty:
    data_cleaned = data_cleaned.drop(columns=cols_to_drop)
    display(
        Markdown(
            f"* **Признаки** (`{', '.join(cols_to_drop)}`) были удалены из-за высокого процента пропущенных значений **(более {THRESHOLD}%)**."
        )
    )
else:
    display(
        Markdown(
            "* Признаков, где процент пропущенных значений превышает установленный порог не найдено."
        )
    )

# Обновление групп признаков после удаления столбцов с высоким процентом пропусков
VALID_FEATURES = {
    k: [col for col in v if col in data_cleaned.columns]
    for k, v in FEATURE_CONFIG.items()
}

# Распаковка групп признаков по типу
numerical_cols, ordinal_cols, binary_cols, nominal_cols = [
    VALID_FEATURES[k] for k in ["numerical", "ordinal", "binary", "nominal"]
]

# Формирование общего списка категориальных признаков
categorical_cols = ordinal_cols + binary_cols + nominal_cols

# 2. Заполнение пропусков для числовых признаков медианными значениями (медианой)
if numerical_cols:
    data_cleaned[numerical_cols] = data_cleaned[numerical_cols].fillna(
        data_cleaned[numerical_cols].median()
    )
    display(
        Markdown(
            f"* **Числовые признаки** (`{', '.join(numerical_cols)}`) заполнены **медианными значениями (медианой)**."
        )
    )

# 3. Заполнение пропусков для категориальных признаков наиболее частыми значениями (модой)
if categorical_cols:
    for col in categorical_cols:
        data_cleaned[col] = data_cleaned[col].fillna(data_cleaned[col].mode()[0])
    display(
        Markdown(
            f"* **Категориальные признаки** (`{', '.join(categorical_cols)}`) заполнены **наиболее частыми значениями (модой)**."
        )
    )

In [ ]:
# --- 6. АНАЛИЗ ПРОПУСКОВ "ПОСЛЕ" ---

display(Markdown("#### **Анализ пропущенных значений ПОСЛЕ обработки**"))
missing_summary_after = get_missing_summary(data_cleaned)
if missing_summary_after["Missing Values"].sum() == 0:
    display(Markdown("Пропущенные значения в данных отсутствуют."))
else:
    display(Markdown("После обработки в данных все еще есть пропущенные значения:"))
    display(missing_summary_after[missing_summary_after["Missing Values"] > 0])

### **1.2.	Обработка дубликатов**

* __Цель:__ Исключить искажение результатов анализа за счет повторяющихся записей.
* __Задачи:__ Найти и удалить полностью дублирующиеся строки или строки, дублирующиеся по ключевым полям.
* __Алгоритм использования:__
  1. __Поиск:__ Выявление дубликатов во всем наборе данных или по заданным столбцам.
  2. __Анализ:__ Просмотр найденных дубликатов для принятия решения.
  3. __Удаление:__ Удаление всех повторяющихся записей, кроме первого или последнего вхождения.

In [ ]:
# --- 1. ПОИСК И АНАЛИЗ ДУБЛИКАТОВ ---

# Формирование списка столбцов для проверки на дублирующиеся строки (исключая 'id')
cols_to_check = [col for col in data_cleaned.columns if col != "id"]

# Маска для всех копий дубликатов
duplicate_mask = data_cleaned.duplicated(subset=cols_to_check, keep=False)

# Подсчет количества дубликатов, которые подлежат удалению
num_duplicates_to_drop = data_cleaned.duplicated(subset=cols_to_check).sum()

# Просмотр найденных дубликатов для принятия решения
if num_duplicates_to_drop > 0:
    display(Markdown("#### **Найденные дублирующиеся строки (все копии):**"))
    display(data_cleaned[duplicate_mask].sort_values(by=cols_to_check))
else:
    display(Markdown("**Дублирующиеся записи не найдены.**"))

In [ ]:
# --- 2. ПРИМЕНЕНИЕ СТРАТЕГИИ: УДАЛЕНИЕ ---

display(Markdown("#### **Применение стратегии удаления дублирующихся записей**"))

if num_duplicates_to_drop > 0:
    # Сохранение размерности данных ДО удаления
    shape_before = data_cleaned.shape

    # Удаление дубликатов и сброс системного индекса Pandas
    data_cleaned = data_cleaned.drop_duplicates(subset=cols_to_check).reset_index(
        drop=True
    )

    # Проверка на наличие дубликатов после удаления
    num_duplicates_after = data_cleaned.duplicated(subset=cols_to_check).sum()

    display(
        Markdown(
            f"* Найдено дублирующихся записей: **{num_duplicates_to_drop}**.\n\n"
            f"* Количество дубликатов после удаления: **{num_duplicates_after}**.\n\n"
            f"* Размер набора данных изменился с **{shape_before[0]}** на **{data_cleaned.shape[0]}** строк.\n\n"
            f"* Размер набора данных ПОСЛЕ удаления: **{data_cleaned.shape[0]} строк и {data_cleaned.shape[1]} столбцов**."
        )
    )

### **1.3.	Обработка выбросов**

* __Цель:__ Уменьшить влияние аномальных значений, которые могут быть следствием ошибок ввода или редких событий, на статистические характеристики и модели.
* __Задачи:__ Определить аномалии и скорректировать их.
* __Алгоритм использования:__
  1. __Обнаружение:__
     * __Визуальные методы:__ Ящик с усами (Boxplot), гистограммы.
     * __Статистические методы:__
       * __Метод межквартильного размаха:__ Выбросами считаются значения, выходящие за пределы [Q1 - 1.5 * IQR, Q3 + 1.5 * IQR].
       * __Метод Z-отклонения:__ Выбросами считаются значения, чей модуль Z-оценки больше 3 (при нормальном распределении).
  2. __Обработка:__
     *  Удаление (если выбросы — результат ошибки).
     *  Замена на граничные значения.
     *  Замена на среднее или медиану.

In [ ]:
# --- 1. ФУНКЦИИ ДЛЯ АНАЛИЗА И ОБРАБОТКИ ВЫБРОСОВ ---


def get_outlier_summary(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Рассчитывает границы по методу IQR, количество и процент выбросов.
    """
    # Вычисление Q1, Q3 и IQR для каждого числового признака
    Q1 = df[columns].quantile(0.25)
    Q3 = df[columns].quantile(0.75)
    IQR = Q3 - Q1

    # Определение границ для выбросов
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Подсчет количества выбросов по каждому числовому признаку
    outliers = (df[columns] < lower_bound) | (df[columns] > upper_bound)
    outlier_counts = outliers.sum()
    outlier_percentage = (outlier_counts / len(df)) * 100

    summary_df = pd.DataFrame(
        {
            "Q1": Q1,
            "Q3": Q3,
            "IQR": IQR,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Outlier Count": outlier_counts,
            "Outlier Percentage (%)": outlier_percentage,
        }
    )
    return summary_df


def plot_outliers(df: pd.DataFrame, columns: list, title_prefix: str):
    """Строит Boxplot для каждого числового признака."""
    plt.figure(figsize=(16, 6))
    sns.boxplot(data=df[columns], palette="Set2")
    plt.title(title_prefix, fontsize=14)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# --- 2. АНАЛИЗ ВЫБРОСОВ ДО ОБРАБОТКИ ---

# Визуализация выбросов с помощью boxplot для каждого числового признака ДО обработки выбросов
plot_outliers(
    data_cleaned,
    numerical_cols,
    "Ящик с усами (Boxplot) для каждого числового признака ДО обработки выбросов",
)

# Вывод статистики по выбросам по каждому числовому признаку
outlier_summary_before = get_outlier_summary(data_cleaned, numerical_cols)
display(Markdown("#### **Статистика по выбросам ДО обработки:**"))
display(outlier_summary_before)
display(
    Markdown(
        f"Размер данных ДО обработки выбросов: **{data_cleaned.shape[0]} строк и {data_cleaned.shape[1]} столбцов**."
    )
)

In [ ]:
# --- 3. ПРИМЕНЕНИЕ СТРАТЕГИИ ОБРАБОТКИ ---

display(Markdown("#### **Применение стратегии обработки выбросов**"))

# Определение порога для обработки выбросов (5% от общего количества данных)
THRESHOLD = 5.0

for col in numerical_cols:
    summary = outlier_summary_before.loc[col]
    if summary["Outlier Count"] == 0:
        continue

    lower = summary["Lower Bound"]
    upper = summary["Upper Bound"]
    percentage = summary["Outlier Percentage (%)"]

    if percentage > THRESHOLD:
        # Замена выбросов на граничные значения для признаков с большим количеством выбросов (>5%)
        data_cleaned[col] = data_cleaned[col].clip(lower=lower, upper=upper)
        display(
            Markdown(
                f"* **{col}:** процент выбросов **более 5%** от общего количества данных ({percentage:.2f}%). Выбросы заменены на **граничные значения**."
            )
        )
    else:
        # Векторизованная замена выбросов на медианные значения для признаков с меньшим количеством выбросов (<=5%)
        median_value = data_cleaned[col].median()
        data_cleaned[col] = np.where(
            (data_cleaned[col] < lower) | (data_cleaned[col] > upper),
            median_value,
            data_cleaned[col],
        )
        display(
            Markdown(
                f"* **{col}:** процент выбросов **менее 5%** от общего количества данных ({percentage:.2f}%). Выбросы заменены на **медианные значения**."
            )
        )

In [ ]:
# --- 4. АНАЛИЗ ВЫБРОСОВ ПОСЛЕ ОБРАБОТКИ ---

# Визуализация выбросов с помощью boxplot для каждого числового признака ПОСЛЕ обработки выбросов
plot_outliers(
    data_cleaned,
    numerical_cols,
    "Ящик с усами (Boxplot) для каждого числового признака ПОСЛЕ обработки выбросов",
)

# Вывод статистики по выбросам по каждому числовому признаку
outlier_summary_after = get_outlier_summary(data_cleaned, numerical_cols)
display(Markdown("#### **Статистика по выбросам ПОСЛЕ обработки:**"))
display(outlier_summary_after)
display(
    Markdown(
        f"Размер данных ПОСЛЕ обработки выбросов: **{data_cleaned.shape[0]} строк и {data_cleaned.shape[1]} столбцов**."
    )
)

In [ ]:
# --- СОХРАНЕНИЕ ДАННЫХ ---

display(Markdown("#### **Сохранение данных**"))

PROCESSED_DATA_PATH = Path("../data/interim")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Сохранение очищенного набора данных в новый CSV файл
file_path = PROCESSED_DATA_PATH / "heart_disease_uci_cleaned.csv"
data_cleaned.to_csv(file_path, index=False)

# Проверка сохранения данных
display(
    Markdown(
        f"* Данные успешно сохранены в файл: `{file_path}`\n\n"
        f"* Итоговый размер набора данных: **{data_cleaned.shape[0]} строк и {data_cleaned.shape[1]} столбцов**."
    )
)

## **2. Преобразование данных**

### **2.1. Масштабирование данных**

* __Цель:__ Привести признаки к единому масштабу, что необходимо для алгоритмов, чувствительных к размерности данных.
* __Задачи:__ Изменить диапазон значений признаков.
* __Алгоритм использования:__
  * __Стандартизация:__ Приведение данных к среднему $= 0$ и стандартному отклонению $= 1$.
    * Формула: $z = \frac{x - \mu}{\sigma}$ (где $\mu$ — среднее, $\sigma$ — стандартное отклонение).
  * __Нормализация:__ Приведение данных к заданному диапазону, обычно $[0, 1]$.
    * Формула: $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$

In [ ]:
# --- 1. АНАЛИЗ МАСШТАБА "ДО" ---

display(Markdown("#### **Статистика числовых признаков ДО масштабирования**"))
display(data_cleaned[numerical_cols].describe().round(2))

# --- 2. ПРИМЕНЕНИЕ СТРАТЕГИИ: СТАНДАРТИЗАЦИЯ ---

# Создание копии очищенного набора данных для дальнейшей стандартизации числовых признаков
data_scaled = data_cleaned.copy()

# Инициализация стандартизатора
scaler = StandardScaler()

# Стандартизация числовых признаков
data_scaled[numerical_cols] = scaler.fit_transform(data_scaled[numerical_cols])

# --- 3. АНАЛИЗ МАСШТАБА "ПОСЛЕ" ---

display(Markdown("#### **Статистика числовых признаков ПОСЛЕ масштабирования**"))
display(data_scaled[numerical_cols].describe().round(4))

### **2.2.	Кодирование категориальных признаков**

* __Цель:__ Преобразовать текстовые (категориальные) данные в числовой формат, понятный алгоритмам машинного обучения.
* __Задачи:__ Заменить метки категорий на числа.
* __Алгоритм использования:__
  * __Порядковое кодирование:__ Каждой уникальной категории присваивается уникальное целое число в соответствии с ее логическим весом. Применяется для признаков, в которых заложен естественный порядок или иерархия (например, "начальная школа", "средняя", "высшая").
  * __Прямое кодирование:__ Создание новых двоичных (0 или 1) столбцов для каждой уникальной категории исходного признака. Подходит для номинальных данных, где категории равнозначны и не имеют порядка (например, цвета, города). Если категорий много, это приводит к значительному увеличению размерности данных.

In [ ]:
# --- 1. АНАЛИЗ КАТЕГОРИЙ "ДО" ---

display(Markdown("#### **Анализ категориальных признаков ДО кодирования**"))

# Подсчет количества уникальных значений в каждом категориальном признаке
unique_values = data_scaled[categorical_cols].nunique()
unique_values_df = pd.DataFrame(
    {
        "Категориальный признак": unique_values.index,
        "Количество уникальных значений": unique_values.values,
        "Тип": [
            (
                "Порядковый/Бинарный"
                if col in ordinal_cols + binary_cols
                else "Номинальный"
            )
            for col in unique_values.index
        ],
    }
)

display(unique_values_df)

In [ ]:
# --- 2. ПРИМЕНЕНИЕ СТРАТЕГИИ КОДИРОВАНИЯ ---

display(Markdown("#### **Применение стратегии кодирования категориальных признаков**"))

# Создание копии стандартизированных данных для дальнейшего кодирования категориальных признаков
data_encoded = data_scaled.copy()

# 1. Кодирование категориальных признаков с двумя категориями (бинарные признаки)
if binary_cols:
    binary_mapping = {"Male": 1, "Female": 0, "True": 1, "False": 0, True: 1, False: 0}
    data_encoded[binary_cols] = data_encoded[binary_cols].replace(binary_mapping)
    display(
        Markdown(
            f"* **Бинарные признаки** (`{', '.join(binary_cols)}`): применено **ручное кодирование** (значения заменены на 1 и 0)."
        )
    )

# 2. Кодирование категориальных порядковых признаков с учетом медицинского контекста
if ordinal_cols:
    ordinal_mappings = {
        "restecg": {"normal": 0, "st-t abnormality": 1, "lv hypertrophy": 2},
        "slope": {"upsloping": 0, "flat": 1, "downsloping": 2},
    }
    for col, mapping in ordinal_mappings.items():
        if col in data_encoded.columns:
            data_encoded[col] = data_encoded[col].map(mapping)
    display(
        Markdown(
            f"* **Порядковые признаки** (`{', '.join(ordinal_cols)}`): применено **порядковое кодирование (Ordinal Encoding)** с учетом медицинского контекста тяжести состояния."
        )
    )

# 3. Прямое кодирование (One-Hot Encoding) для оставшихся категориальных признаков без порядка (номинальные признаки)
if nominal_cols:
    data_encoded = pd.get_dummies(
        data_encoded, columns=nominal_cols, drop_first=True, dtype=int
    )
    display(
        Markdown(
            f"* **Номинальные признаки** (`{', '.join(nominal_cols)}`): применено **прямое кодирование (One-Hot Encoding)**."
        )
    )

In [ ]:
# --- 3. АНАЛИЗ КАТЕГОРИЙ "ПОСЛЕ" ---

display(Markdown("#### **Содержание набора данных ПОСЛЕ кодирования**"))
display(data_encoded.head())

# Результат изменения
display(
    Markdown(
        f"##### **Результат изменения размерности данных**\n"
        f"* Размер ДО: **{data_scaled.shape[1]}** столбцов.\n"
        f"* Размер ПОСЛЕ: **{data_encoded.shape[1]}** столбцов.\n"
        f"* Размер данных ПОСЛЕ кодирования: **{data_encoded.shape[0]} строк и {data_encoded.shape[1]} столбцов**."
    )
)

### **2.3.	Трансформация распределения**

* __Цель:__ Приблизить распределение данных к нормальному, что требуется для некоторых статистических тестов и моделей (линейная регрессия).
* __Задачи:__ Уменьшить асимметрию распределения.
  * __Алгоритм использования:__
    * __Логарифмирование:__ Применяется к положительным числам для сжатия длинного хвоста распределения.
    * __Степенные преобразования:__ Более общий подход. Метод Бокса-Кокса применяется для строго положительных данных. Метод Йео-Джонсона поддерживает работу как с положительными, так и с отрицательными значениями

In [ ]:
# --- 1. ФУНКЦИИ ДЛЯ АНАЛИЗА РАСПРЕДЕЛЕНИЯ ---


def get_distribution_stats(
    df: pd.DataFrame, columns: list[str], suffix: str, alpha: float = 0.05
) -> pd.DataFrame:
    """Рассчитывает асимметрию (skewness) для числовых признаков и проводит Shapiro-Wilk тест для проверки нормальности распределения."""
    results = []
    for col in columns:
        skew_val = df[col].skew()
        stat, p_value = shapiro(df[col])
        results.append(
            {
                "Признак": col,
                "Статистика": stat,
                f"Асимметрия {suffix} преобразования": skew_val,
                "p-value": p_value,
                f"Нормальное (p > {alpha})": p_value > alpha,
            }
        )
    return pd.DataFrame(results)

In [ ]:
# --- 2. АНАЛИЗ "ДО" ПРЕОБРАЗОВАНИЯ ---

display(Markdown("#### **Анализ распределения ДО преобразования**"))
stats_before_df = get_distribution_stats(data_encoded, numerical_cols, "ДО")
display(stats_before_df)

In [ ]:
# --- 3. ПРИМЕНЕНИЕ СТРАТЕГИИ ТРАНСФОРМАЦИИ РАСПРЕДЕЛЕНИЯ ---

# Инициализация PowerTransformer с методом Yeo-Johnson, который может обрабатывать как положительные, так и отрицательные значения
pt = PowerTransformer(method="yeo-johnson", standardize=True)

# Применение PowerTransformer к числовым признакам
data_transformed = data_encoded.copy()
data_transformed[numerical_cols] = pt.fit_transform(data_transformed[numerical_cols])

In [ ]:
# --- 4. АНАЛИЗ "ПОСЛЕ" ПРЕОБРАЗОВАНИЯ ---

display(
    Markdown("#### **Сравнительный анализ распределения ДО и ПОСЛЕ преобразования**")
)
stats_after_df = get_distribution_stats(data_transformed, numerical_cols, "ПОСЛЕ")

# Объединение статистик для сравнения
comparison_df = pd.merge(
    stats_before_df, stats_after_df, on="Признак", suffixes=(" ДО", " ПОСЛЕ")
)

# Вычисление метрики уменьшения асимметрии
comparison_df["Уменьшение асимметрии"] = abs(
    comparison_df["Асимметрия ДО преобразования"]
) - abs(comparison_df["Асимметрия ПОСЛЕ преобразования"])

columns_to_show = [
    "Признак",
    "Асимметрия ДО преобразования",
    "Асимметрия ПОСЛЕ преобразования",
    "Уменьшение асимметрии",
    "p-value ДО",
    "p-value ПОСЛЕ",
]

display(comparison_df[columns_to_show])

In [ ]:
# --- 5. ВИЗУАЛИЗАЦИЯ РАСПРЕДЕЛЕНИЯ "ДО" и "ПОСЛЕ" ПРЕОБРАЗОВАНИЯ ---

display(
    Markdown("#### **Распределения числовых признаков ДО и ПОСЛЕ преобразования:**")
)

plt.figure(figsize=(16, 8))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(2, 3, i)
    sns.kdeplot(data_encoded[col], fill=True, color="skyblue", label="ДО")
    sns.kdeplot(
        data_transformed[col],
        fill=True,
        color="lightcoral",
        label="ПОСЛЕ (Yeo-Johnson)",
    )
    plt.title(f"Распределение: {col}")
    plt.xlabel(f"{col}")
    plt.ylabel("Плотность (Density)")
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- СОХРАНЕНИЕ ДАННЫХ ---

display(Markdown("#### **Сохранение данных**"))

PROCESSED_DATA_PATH = Path("../data/processed")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Сохранение преобразованного набора данных в новый CSV файл
file_path = PROCESSED_DATA_PATH / "heart_disease_uci_processed.csv"
data_transformed.to_csv(file_path, index=False)

display(
    Markdown(
        f"* Данные успешно сохранены в файл: `{file_path}`\n\n"
        f"* Итоговый размер набора данных: **{data_transformed.shape[0]} строк и {data_transformed.shape[1]} столбцов**."
    )
)